# B2: EfficientNet-B3 — 5-Fold Cross-Validation.

**Author:** Mridankan Mandal.

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass).


## Overview.

This notebook trains an **EfficientNet-B3** model for multi-target biomass regression
using **5-fold Stratified Group K-Fold** cross-validation on the training set.

### Architecture.
- **Backbone:** EfficientNet-B3 (ImageNet pretrained via torchvision).
- **Head:** FC(1536→512) → ReLU → Dropout(0.3) → FC(512→5).
- **Loss:** Weighted MSE (competition weights: 0.1, 0.1, 0.1, 0.2, 0.5).
- **Cross-Validation:** 5-fold Stratified Group K-Fold (same splits as B4/proposed model).


In [ ]:
from PIL import Image
from IPython.display import display

img = Image.open("/kaggle/input/image12/biomass pic.png")
display(img)


In [ ]:
# ============================================================
# B2: EfficientNet-B3 — 5-Fold Stratified Group K-Fold CV
# ============================================================

import os, gc, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import StratifiedGroupKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

# -------------------------
# Config
# -------------------------
DATA_DIR = "/kaggle/input/competitions/csiro-biomass"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 17
BATCH_SIZE = 16
EPOCHS = 50
IMG_SIZE = 384
LR = 1e-4
NUM_WORKERS = 2
N_FOLDS = 5
EARLY_STOPPING_PATIENCE = 10

TARGET_NAMES = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
COMP_WEIGHTS_DICT = {"Dry_Green_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Clover_g": 0.1, "GDM_g": 0.2, "Dry_Total_g": 0.5}
COMP_WEIGHTS = np.array([COMP_WEIGHTS_DICT[t] for t in TARGET_NAMES])

random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

# -------------------------
# Load & Pivot
# -------------------------
train_long = pd.read_csv(TRAIN_CSV)
train_long['image_id'] = train_long['sample_id'].str.split('__').str[0]

pivot = train_long.pivot_table(
    index=['image_id', 'image_path'],
    columns='target_name',
    values='target',
    aggfunc='first'
).reset_index()

for col in TARGET_NAMES:
    if col not in pivot.columns:
        pivot[col] = 0.0

def make_full_path(p):
    if os.path.isabs(p):
        return p
    return os.path.join(DATA_DIR, p)

pivot["image_full_path"] = pivot["image_path"].apply(make_full_path)

exists_mask = pivot["image_full_path"].apply(os.path.exists)
if not exists_mask.all():
    print(f"Warning: {(~exists_mask).sum()} missing image files dropped.")
    pivot = pivot.loc[exists_mask].reset_index(drop=True)

# -------------------------
# Fold Splits (same as B4/proposed)
# -------------------------
pivot['total_bin'] = pd.qcut(pivot['Dry_Total_g'], q=5, labels=False, duplicates='drop')

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
pivot['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(pivot, pivot['total_bin'], groups=pivot['image_id'])):
    pivot.loc[val_idx, 'fold'] = fold

print(f"Training images: {len(pivot)}")
print(f"Fold distribution:\n{pivot['fold'].value_counts().sort_index()}")
display(pivot.head())

# -------------------------
# Dataset & Transforms
# -------------------------
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BiomassImageDataset(Dataset):
    def __init__(self, df, target_cols=None, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.target_cols = target_cols if target_cols is not None else []

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_full_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        targets = row[self.target_cols].values.astype(np.float32)
        return img, torch.tensor(targets, dtype=torch.float32)

# -------------------------
# Model
# -------------------------
class BiomassModel(nn.Module):
    def __init__(self, num_targets=5, pretrained_weights=True):
        super().__init__()
        weights = EfficientNet_B3_Weights.DEFAULT if pretrained_weights else None
        self.backbone = efficientnet_b3(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_targets)
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

# -------------------------
# Loss & Metrics
# -------------------------
mse_loss = nn.MSELoss(reduction="none")
weights_tensor = torch.tensor(COMP_WEIGHTS, dtype=torch.float32).to(DEVICE)

def weighted_mse_loss(preds, targets):
    loss_per_elem = mse_loss(preds, targets)
    mean_per_target = loss_per_elem.mean(dim=0)
    return (mean_per_target * weights_tensor).sum()

def weighted_r2_score(y_true, y_pred):
    r2_scores = []
    for i in range(y_true.shape[1]):
        yt, yp = y_true[:, i], y_pred[:, i]
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    r2_scores = np.array(r2_scores)
    weighted = np.sum(r2_scores * COMP_WEIGHTS) / np.sum(COMP_WEIGHTS)
    return weighted, r2_scores

# -------------------------
# Train / Validate functions
# -------------------------
def train_one_epoch(model, loader, optimizer, scaler=None):
    model.train()
    running_loss, n = 0.0, 0
    for imgs, targets in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                preds = model(imgs)
                loss = weighted_mse_loss(preds, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(imgs)
            loss = weighted_mse_loss(preds, targets)
            loss.backward()
            optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
    return running_loss / n

def validate(model, loader):
    model.eval()
    preds_list, trues_list = [], []
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Val", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            out = model(imgs)
            preds_list.append(out.cpu().numpy())
            trues_list.append(targets.numpy())
    preds = np.vstack(preds_list)
    trues = np.vstack(trues_list)
    w_r2, per_target = weighted_r2_score(trues, preds)
    return w_r2, per_target

# -------------------------
# 5-Fold CV Training Loop
# -------------------------
fold_results = []

for fold in range(N_FOLDS):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    train_fold = pivot[pivot['fold'] != fold].reset_index(drop=True)
    val_fold = pivot[pivot['fold'] == fold].reset_index(drop=True)
    print(f"Train: {len(train_fold)}, Val: {len(val_fold)}")

    train_ds = BiomassImageDataset(train_fold, target_cols=TARGET_NAMES, transform=train_tfm)
    val_ds = BiomassImageDataset(val_fold, target_cols=TARGET_NAMES, transform=val_tfm)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = BiomassModel(num_targets=len(TARGET_NAMES)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    best_w_r2 = -1e9
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler=scaler)
        w_r2, per_r2 = validate(model, val_loader)

        improved = ""
        if w_r2 > best_w_r2:
            best_w_r2 = w_r2
            torch.save(model.state_dict(), f"best_model_fold{fold}.pth")
            patience_counter = 0
            improved = " *"
        else:
            patience_counter += 1

        print(f"  Epoch {epoch}/{EPOCHS} | Loss: {train_loss:.6f} | W-R²: {w_r2:.6f}{improved}")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

    # Evaluate best checkpoint
    model.load_state_dict(torch.load(f"best_model_fold{fold}.pth", weights_only=True))
    w_r2, per_r2 = validate(model, val_loader)
    fold_results.append({'fold': fold, 'weighted_r2': w_r2, 'per_target_r2': per_r2})

    print(f"\n  Fold {fold} Best Weighted R²: {w_r2:.6f}")
    for j, col in enumerate(TARGET_NAMES):
        print(f"    {col}: R² = {per_r2[j]:.6f}")

    del model, optimizer, scaler, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

# -------------------------
# Summary
# -------------------------
w_r2_all = [r['weighted_r2'] for r in fold_results]
per_target_all = np.array([r['per_target_r2'] for r in fold_results])

print(f"\n{'='*60}")
print(f"B2: EfficientNet-B3 — {N_FOLDS}-Fold CV Results")
print(f"{'='*60}")
print(f"Mean Weighted R²: {np.mean(w_r2_all):.6f} ± {np.std(w_r2_all):.6f}")
print()
for j, col in enumerate(TARGET_NAMES):
    print(f"  {col}: {np.mean(per_target_all[:, j]):.6f} ± {np.std(per_target_all[:, j]):.6f}")


## Conclusion.

This notebook trains EfficientNet-B3 with 5-fold Stratified Group K-Fold cross-validation
on the training set, using the same fold splits as B4 and the proposed model for fair comparison.

Key design choices:
- **Weighted MSE loss** aligned with competition evaluation weights.
- **Early stopping** (patience=10) to prevent overfitting.
- **Per-fold best checkpoint** evaluated for final metrics.
